In [6]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import os

In [2]:
input_csv = 'Preprocessed_pubmed_data.csv'  # Change if your file name is different

In [3]:
input_csv

'Preprocessed_pubmed_data.csv'

In [4]:
graphml_output = 'food_health_knowledge_graph.graphml'
plot_output = 'kg_preview.png'
triples_csv = 'pubmed_triples.csv'

In [7]:
if not os.path.exists(input_csv):
    raise FileNotFoundError(f"Input file {input_csv} not found. Please check the path.")

df = pd.read_csv(input_csv)

print(f"Loaded {len(df)} rows from {input_csv}")
print("Columns:", df.columns.tolist())

Loaded 30567 rows from Preprocessed_pubmed_data.csv
Columns: ['ingredient', 'pmid', 'title', 'year', 'journal', 'diseases_mentioned', 'mesh_disease_terms', 'relationship_type', 'key_findings', 'authors', 'pubmed_url']


In [8]:
COMMON_DISEASES = [
    'diabetes', 'obesity', 'hypertension', 'cardiovascular disease', 'cancer', 'alzheimer',
    'dementia', 'osteoporosis', 'chronic kidney disease', 'kidney disease', 'lung cancer',
    'inflammation', 'insulin resistance', 'gestational diabetes', 'abdominal obesity',
    'overweight', 'heart disease', 'stroke', 'metabolic syndrome', 'allergic',
    'alzheimer disease', 'type 2 diabetes'
]

In [9]:
def extract_diseases(row):
    diseases = set()
    
    # Collect from mesh_disease_terms if available
    if pd.notna(row['mesh_disease_terms']) and str(row['mesh_disease_terms']).strip():
        terms = [t.strip().lower() for t in str(row['mesh_disease_terms']).split(',')]
        diseases.update(terms)
    
    # ALSO collect from diseases_mentioned if available (equal treatment - no priority)
    if pd.notna(row['diseases_mentioned']) and str(row['diseases_mentioned']).strip():
        terms = [t.strip().lower() for t in str(row['diseases_mentioned']).split(',')]
        diseases.update(terms)
    
    # Only if BOTH structured fields are empty → fallback to key_findings
    if not diseases:
        findings = str(row['key_findings']).lower()
        for disease in COMMON_DISEASES:
            if disease in findings:
                diseases.add(disease)
        
        # Regex for common variants/phrases
        patterns = [
            r'\b(type 2 diabetes|gestational diabetes|alzheimer.?s?.? disease|lung cancer|chronic kidney disease)\b',
            r'\b(cardiovascular disease|heart disease|kidney disease|alzheimer disease)\b'
        ]
        for pattern in patterns:
            matches = re.findall(pattern, findings, re.IGNORECASE)
            diseases.update([m.lower() for m in matches])
    
    return list(diseases)

In [10]:
df['disease_list'] = df.apply(extract_diseases, axis=1)

In [11]:
original_rows = len(df)
df = df[df['disease_list'].map(len) > 0].reset_index(drop=True)
print(f"Rows after disease extraction: {len(df)} (dropped {original_rows - len(df)} rows with no diseases)")

Rows after disease extraction: 30567 (dropped 0 rows with no diseases)


In [12]:
df_exploded = df.explode('disease_list').reset_index(drop=True)
df_exploded['disease'] = df_exploded['disease_list'].str.lower().str.strip()

In [13]:
df_exploded['disease'] 

0                      diabetes
1        cardiovascular disease
2            insulin resistance
3                     alzheimer
4                      diabetes
                  ...          
67017              experimental
67018                    type 2
67019         diabetes mellitus
67020                  diabetes
67021           type 2 diabetes
Name: disease, Length: 67022, dtype: object

In [14]:
df_exploded['ingredient'] = df_exploded['ingredient'].str.lower().str.strip()

print(f"Final ingredient-disease triples: {len(df_exploded)}")

Final ingredient-disease triples: 67022


In [15]:
G = nx.MultiDiGraph()

In [16]:
for _, row in df_exploded.iterrows():
    ingredient = row['ingredient']
    disease = row['disease']
    pmid = str(row['pmid'])
    
    # Add nodes with properties
    G.add_node(ingredient, node_type='Ingredient')
    G.add_node(disease, node_type='Disease')
    G.add_node(pmid, node_type='Publication',
               title=row.get('title', ''),
               year=row.get('year', ''),
               journal=row.get('journal', ''),
               authors=row.get('authors', ''),
               key_findings=row.get('key_findings', ''),
               pubmed_url=row.get('pubmed_url', ''))
    
    # Main association edge
    G.add_edge(ingredient, disease,
               key='ASSOCIATED_WITH',
               relationship_type=row.get('relationship_type', 'Unknown'),
               key_findings=row.get('key_findings', ''),
               evidence_pmid=pmid)
    
    # Evidence edges for traceability
    G.add_edge(ingredient, pmid, key='SUPPORTED_BY')
    G.add_edge(pmid, disease, key='EVIDENCE_FOR')

In [17]:
print(f"\nKnowledge Graph built successfully!")
print(f"   Nodes: {G.number_of_nodes()}")
print(f"     - Ingredients: {len([n for n,d in G.nodes(data=True) if d.get('node_type')=='Ingredient'])}")
print(f"     - Diseases: {len([n for n,d in G.nodes(data=True) if d.get('node_type')=='Disease'])}")
print(f"     - Publications: {len([n for n,d in G.nodes(data=True) if d.get('node_type')=='Publication'])}")
print(f"   Edges: {G.number_of_edges()}")


Knowledge Graph built successfully!
   Nodes: 25070
     - Ingredients: 224
     - Diseases: 287
     - Publications: 24559
   Edges: 97316


In [18]:
preview_limit = 500

print(f"PREVIEW: First {preview_limit} extracted triples (for verification)\n")
preview = df_exploded[['ingredient', 'disease', 'relationship_type', 'pmid', 'key_findings']].head(preview_limit).copy()
preview['key_findings'] = preview['key_findings'].str[:120] + '...'  # Truncate long text
preview = preview.rename(columns={
    'ingredient': 'Ingredient',
    'disease': 'Disease',
    'relationship_type': 'Relationship Type',
    'pmid': 'Evidence PMID',
    'key_findings': 'Key Findings Snippet'
})

PREVIEW: First 500 extracted triples (for verification)



In [19]:
print(preview.to_string(index=False))
print("\n" + "="*80)
print("If the preview looks correct, the script will continue to build and save the graph.")
print("If not, stop the script and adjust the data/extraction logic.")
print("="*80 + "\n")

Ingredient                           Disease     Relationship Type  Evidence PMID                                                                                                        Key Findings Snippet
   protein                          diabetes           Risk Factor       41390803 The results indicated that elevated TyG-ABSI values were positively associated with the risk of ASCVD in the population ...
   protein            cardiovascular disease           Risk Factor       41390803 The results indicated that elevated TyG-ABSI values were positively associated with the risk of ASCVD in the population ...
   protein                insulin resistance           Risk Factor       41390803 The results indicated that elevated TyG-ABSI values were positively associated with the risk of ASCVD in the population ...
   protein                         alzheimer           Risk Factor       41390778 We analyzed electronic health records from UCLA Health System (N = 416,212; genetic subset N =

In [20]:
triples_columns = ['ingredient', 'disease', 'relationship_type', 'pmid', 'title', 'year', 'journal', 'authors', 'key_findings', 'pubmed_url']
df_exploded.to_csv(triples_csv, index=False, columns=triples_columns)
print(f"Final triples saved to: {triples_csv} ({len(df_exploded)} rows)")

Final triples saved to: pubmed_triples.csv (67022 rows)


In [21]:
G = nx.MultiDiGraph()
for _, row in df_exploded.iterrows():
    ing = row['ingredient']
    dis = row['disease']
    pmid = str(row['pmid'])
    G.add_node(ing, node_type='Ingredient')
    G.add_node(dis, node_type='Disease')
    G.add_node(pmid, node_type='Publication', title=row.get('title',''), year=row.get('year',''), key_findings=row.get('key_findings',''))
    G.add_edge(ing, dis, key='ASSOCIATED_WITH', relationship_type=row.get('relationship_type','Unknown'), evidence_pmid=pmid)
    G.add_edge(ing, pmid, key='SUPPORTED_BY')
    G.add_edge(pmid, dis, key='EVIDENCE_FOR')

nx.write_graphml(G, graphml_output)
print(f"NetworkX graph saved to: {graphml_output}")

NetworkX graph saved to: food_health_knowledge_graph.graphml
